# RADON — demo

Robust autoencoder for unsupervised anomaly detection.

This notebook contains **no logic**: every step calls the `radon` package. It exists to show
the pipeline and its plots, not to implement anything. For a full run use the CLI:

```bash
python -m radon.cli.train --config configs/mnist.yaml
```


## 1. Configuration

Settings come from `configs/default.yaml`, overridden by a dataset file, overridden by
explicit `--set`-style strings. Nothing is hardcoded here.


In [ ]:
from radon.config import load_config
from radon.logging_utils import setup_logging
from radon.seeding import set_seed
from radon.device import resolve_device

cfg = load_config(
    "../configs/mnist.yaml",
    overrides=[
        "training.epochs=40",
        "training.warmup_epochs=20",
        "training.projection_every_warmup=5",
        "training.projection_every_nl=5",
    ],
)

setup_logging(cfg.logging, cfg.paths)
set_seed(cfg.seed, deterministic=cfg.deterministic)
device = resolve_device(cfg.device)
device

## 2. Data

One digit is normal; the training set is contaminated with a few samples of other digits.
The builder validates shapes, dtypes and label values before returning.


In [ ]:
from radon.data import build_dataset

dataset = build_dataset(cfg)
print(dataset.describe())

In [ ]:
from radon.viz import plot_label_distribution

plot_label_distribution(dataset.train_stats, title="Training split")
plot_label_distribution(dataset.test_stats, title="Test split");

## 3. Model


In [ ]:
from radon.models import build_model

model = build_model(cfg.model, input_dim=dataset.n_features).to(device)
model

## 4. Training

Warm-up teaches reconstruction; negative learning then pushes suspected outliers towards
their feature-wise inverse. Periodic **projections** re-partition the data into `L` and `S`.

The trainer returns structured records — no log parsing involved.


In [ ]:
from radon.training import RadonTrainer

trainer = RadonTrainer(model=model, cfg=cfg.training, device=device, compat=cfg.compat)
history = trainer.train(dataset.x_train, feature_names=dataset.feature_names)

print(f"{len(history.epochs)} epochs, {len(history.projections)} projections")
print(f"best epoch: {history.best_epoch}")

In [ ]:
import pandas as pd

pd.DataFrame([vars(r) for r in history.projections])[
    ["epoch", "n_bins", "threshold_inlier", "threshold_outlier", "n_inliers", "n_outliers", "lambda_eff"]
]

The total loss **steps up** when the first projection finds outliers: the objective gains a
term it did not have before. That is expected, not a failure — which is why the inlier term
is recorded separately.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
epochs = [r.epoch for r in history.epochs]
ax.plot(epochs, [r.loss_mean for r in history.epochs], label="total")
ax.plot(epochs, [r.loss_inlier_mean for r in history.epochs], label="inlier term")
ax.axvline(cfg.training.warmup_epochs, color="red", ls="--", label="negative learning starts")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.legend(); ax.grid(alpha=0.3);

## 5. Evaluation

The anomaly score is `1 - cos(f(x), x)`. The decision threshold comes from the last
projection, so it belongs to the model being evaluated.


In [ ]:
from radon.data import build_test_loader
from radon.evaluation import score_split, evaluate_split

_, threshold = history.final_thresholds()
loader = build_test_loader(dataset, batch_size=cfg.data.batch_size, device=device)
scored = score_split(model, loader, device, threshold_cosine=threshold)
result = evaluate_split(scored, legacy_auprc=cfg.compat.legacy_auprc_trapezoid)

print(result.describe())

In [ ]:
from radon.viz.evaluation import plot_score_distribution, plot_roc, plot_precision_recall

plot_score_distribution(scored)
plot_roc(result)
plot_precision_recall(result);

Note the gap between the F1 at the training threshold and the best F1 on the curve. The
projection threshold is tuned to select *confident* outliers for the training loss, not to
classify — it is deliberately conservative, and costs recall when reused as a decision rule.


## 6. What the model looked at


In [ ]:
from radon.viz.reconstruction import collect_reconstructions, plot_reconstructions

inliers, outliers = collect_reconstructions(
    model, loader, device, cfg.data.mnist.image_shape, n_per_class=3
)
plot_reconstructions(inliers, title="Inliers")
plot_reconstructions(outliers, title="Outliers");

For NSL-KDD, `radon.explain` attributes each anomaly to specific features and
categorical groups by ablation. See `python -m radon.cli.explain --config configs/nsl_kdd.yaml`.
